## Part A — Databricks setup and Bronze layer
1. Create a schema named `retail_fresher`.
2. Create a managed volume named `retail_raw`.
3. Upload the three CSV files into the volume.

In [0]:
import shutil
import os
import pyspark.sql.functions as F

In [0]:
%sql
USE CATALOG retail_pulse;

CREATE SCHEMA IF NOT EXISTS retail_fresher
COMMENT 'This schema holds raw files';

CREATE SCHEMA IF NOT EXISTS retail_bronze
COMMENT 'This schema holds bronze tables';

CREATE SCHEMA IF NOT EXISTS retail_silver
COMMENT 'This schema holds silver tables';

CREATE SCHEMA IF NOT EXISTS retail_gold
COMMENT 'This schema holds gold tables';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail_fresher.retail_raw
COMMENT 'This volume holds raw files';

In [0]:
VOLUME_PATH = "/Volumes/retail_pulse/retail_fresher/retail_raw"
BRONZE_SCHEMA = "retail_pulse.retail_bronze"
SILVER_SCHEMA = "retail_pulse.retail_silver"
GOLD_SCHEMA = "retail_pulse.retail_gold"

In [0]:
source_path = "/Workspace/Users/madhulikasawant@gmail.com/retail-pulse-weekly-sales-intelligence/datasets"
file_names = ["customers_500.csv", "products_500.csv", "sales_orders_500.csv"]

for file_name in file_names:
    source_file = os.path.join(source_path, file_name)
    destination_file = os.path.join(VOLUME_PATH, file_name)
    
    print(f"Copying {source_file} to {destination_file}")
    shutil.copy(source_file, destination_file)

display(dbutils.fs.ls(VOLUME_PATH))

4. Read every CSV with PySpark.
5. Keep the initial CSV columns as strings in the Bronze layer.
6. Add `source_file` and `ingestion_timestamp`.
7. Save managed Delta tables:
   - `bronze_customers`
   - `bronze_products`
   - `bronze_sales_orders`


In [0]:
def read_raw_file(file_name):
    file_path = os.path.join(VOLUME_PATH, file_name)
    df = spark.read.option("header", "True").option("inferSchema", False).csv(file_path)
    df_with_metadata = df.withColumns(
        {"source_file": F.lit(file_path), "ingestion_timestamp": F.current_timestamp()}
    )
    return df_with_metadata

def ingest_bronze_data(file_name, table_name):
    raw_df = read_raw_file(file_name)
    raw_df_row_count = raw_df.count()

    raw_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    print(f"Loaded {raw_df_row_count} records into {table_name}")

    bronze_df = spark.table(f"{BRONZE_SCHEMA}.{table_name}")
    bronze_df_row_count = bronze_df.count()

    assert (
        raw_df_row_count == bronze_df_row_count
    ), f"Expected {raw_df_row_count} records in {table_name}, found {bronze_df_row_count}"

    assert [(f.name, f.dataType) for f in raw_df.schema] == [
        (f.name, f.dataType) for f in bronze_df.schema
    ], f"Expected schema {raw_df.schema} in {table_name}, found {bronze_df.schema}"

    return bronze_df

bronze_customers_df = ingest_bronze_data("customers_500.csv", "bronze_customers")
bronze_products_df = ingest_bronze_data("products_500.csv", "bronze_products")
bronze_sales_orders_df = ingest_bronze_data("sales_orders_500.csv", "bronze_sales_orders")
